# The REG731_EVENT_DATA table 

The `REG731_EVENT_DATA` table is a key component of the **Unitary Patent (UP)** module within the PATSTAT EP Register database. Its purpose is to record legal events that specifically affect patents with unitary effect. In simple terms, this table works like a chronological logbook of all legally relevant actions that occur after a patent has obtained unitary effect.

While `REG301_EVENT_DATA` covers legal events for all European patent applications, `REG731_EVENT_DATA` focuses exclusively on applications that have entered the UP phase.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG731_EVENT_DATA
from sqlalchemy import select, func, case, select, and_

patstat = PatstatClient(env='TEST')

db = patstat.orm()

## Key fields in REG731_EVENT_DATA table

### ID (primary key)
A technical identifier for an application that has entered the **UP phase**, without business meaning. Its values will not change from one PATSTAT edition to the next.

### EVENT DATE
The ``EVENT_DATE`` attribute refers to the date when the event was recorded in the file. This date may differ from the effective legal or filing date of the event itself, as it marks when the action was officially entered into the system. The ``EVENT_DATE`` helps track the timing of the event within the EPO's records, providing an important reference point for understanding the sequence of events in the patent application process.

### EVENT_CODE
This attribute represents the internal code used by the EPO to identify the type of legal action or event related to a patent application. This code is essential for classifying and tracking the various actions, such as publications, changes, or deletions, that occur throughout the patent process. The code is up to 30 characters long and may include descriptive elements, such as "9199" for actions before B1 publication or "9299" for actions after B1 publication. It helps ensure consistent identification and understanding of each specific event.

In [2]:
q = (
    db.query(
        REG731_EVENT_DATA.id,
        REG731_EVENT_DATA.event_date,
        REG731_EVENT_DATA.event_code
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,event_date,event_code
0,18711816,2023-12-08,0009798UPPR
1,18795310,2023-11-17,0009798UPPR
2,17927093,2023-12-01,0009799UPPR
3,19759485,2023-12-01,0009799UPPR
4,18721115,2024-02-02,0009799UPPR
...,...,...,...
630,21154246,2024-02-02,0009703RFEE22
631,21200465,2024-02-02,0009703RFEE22
632,21204068,2024-01-12,0009703RFEE22
633,22725234,2024-02-09,0009703RFEE22


## European Patent Bullettin attributes

### BULLETIN_YEAR 
In the PATSTAT database, the ``BULLETIN_YEAR`` field captures the year when an action or event related to a patent application was published in the European Patent Bulletin. This field plays a critical role in tracking the timeline of patent events, ensuring chronological accuracy in analyses.

The ``BULLETIN_YEAR`` is a 4-digit numeric field (formatted as YYYY), with a default value of 0 to indicate cases where no European Patent Bulletin publication is known. For entries where publication in the European Patent Bulletin is confirmed, ``BULLETIN_YEAR`` reflects the corresponding year of publication. It is used in conjunction with ``BULLETIN_NR``, which specifies the European Patent Bulletin issue number.

### BULLETIN_NR

The ``BULLETIN_NR`` attribute represents the issue number of the European Patent Bulletin in which a specific action has been published. This number indicates the calendar week during which the European Patent Bulletin was released. It serves as a reference for identifying the exact edition of the European Patent Bulletin where actions such as patent grants, publications, or other significant events are announced. If the action was not published in the European Patent Bulletin or if the information is unknown, the default value of 0 is used for the ``BULLETIN_NR``, which corresponds to the absence of a known European Patent Bulletin number. This value is only used when the associated ``BULLETIN_YEAR`` is also set to 0.

### BULLETIN_DATE
The ``BULLETIN_DATE`` attribute refers to the date when the event was published in the European Patent Bulletin. For events that are also published in the Official Journal of the European Patent Office (OJ EPO) or European Patent Bulletin, this date marks the public disclosure of the event. If the event has not been published in the European Patent Bulletin, the default value is set to 9999-12-31. This attribute provides a reference for understanding when legal events are communicated publicly, helping to track the dissemination of patent-related information.

In [3]:
q = (
    db.query(
        REG731_EVENT_DATA.id,
        REG731_EVENT_DATA.bulletin_nr,
        REG731_EVENT_DATA.bulletin_date,
        REG731_EVENT_DATA.bulletin_year
    )
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,bulletin_nr,bulletin_date,bulletin_year
0,18711816,2,2024-01-10,2024
1,18795310,51,2023-12-20,2023
2,17927093,1,2024-01-03,2024
3,19759485,1,2024-01-03,2024
4,18721115,10,2024-03-06,2024
...,...,...,...,...
481,21154246,0,9999-12-31,0
482,21200465,0,9999-12-31,0
483,21204068,0,9999-12-31,0
484,22725234,0,9999-12-31,0


## Linking REG701_APPLN to REG731_EVENT_DATA
By performing a join using the `ID` field, it is possible to enrich unitary phase data with the chronological log of legal events stored in the `REG731_EVENT_DATA` table. This join makes it possible to link each record in `REG701_APPLN` to the full sequence of legally relevant actions that occurred during the life cycle of the UP.

In [7]:
from epo.tipdata.patstat.database.models import REG701_APPLN

q = (
    db.query(
        REG731_EVENT_DATA.id,
        REG701_APPLN.status,
        REG731_EVENT_DATA.event_date,
        REG731_EVENT_DATA.event_code
    )
    .join(
        REG701_APPLN,
        REG701_APPLN.id == REG731_EVENT_DATA.id
    )
    .filter(REG731_EVENT_DATA.id == 21154246)
    .distinct()
)

res = patstat.df(q)
res

,id,status,event_date,event_code
0,21154246,9,2023-10-13,0009700UREQ01
1,21154246,9,2023-10-13,0009701UREQ02
2,21154246,9,2024-02-02,0009703RFEE22
